### Advanced Window Functions & Aggregations (Interview Level)

SF Fire Department Data Analysis

#### The Business Request

##### Request:  

“We need to identify our top-performing neighborhoods for emergency response across the city. I want to see the top 3 neighborhoods for each type of emergency call.

We’re specifically interested in response efficiency – neighborhoods that respond quickly, handle high call volumes, and perform consistently.

##### Can you also show me:

- How each neighborhood compares to others in their battalion

- Performance trends over time

- Year-over-year improvements

- Statistical distributions (percentiles)

- Summary rollups by district for executive reporting

#### Step 1 : Clarify The Requirments 

Before jumping into code, a strong candidate asks calrifying questions to avoid rework and align with business needs.

##### Questions You Should Ask:

###### 1. Define Efficiency : 
- “How should we measure efficiency? Is it purely response time, or should we factor in call volume and consistency?”
- ✅ Expected Answer: “Combine multiple factors: fast response (primary), ability to handle volume, and consistent performance.”

###### 2. Time Period & Granularity

- “What time period should we analyze? What granularity - daily, weekly, monthly?”
- ✅ Expected Answer: “Use weekly granularity to smooth out daily noise while keeping recent trends visible.”

###### 3. Statistical Metrics

- “When you say ‘percentiles,’ which ones are most important? 50th (median), 95th, 99th?”
- ✅ Expected Answer: “Show 25th, 50th (median), 75th, and 95th to understand the full distribution.”

###### 4. Moving Average Window

- “For trends over time, what window size for moving averages - 4 weeks, 8 weeks, 12 weeks?”
- ✅ Expected Answer: “7 weeks gives good balance between responsiveness and smoothing.”

###### 6. Battalion Comparison  
- “What is a ‘battalion’ in this context, and why compare within battalions?”
- ✅ Expected Answer: “Battalion is a fire department command district. Compare within battalions because they face similar geographic and operational conditions – makes comparison fair.”

###### 7. Output Format  
- “How should district rollups be presented? Do you need subtotals at multiple levels?”
- ✅ Expected Answer: “Yes, hierarchical rollups: neighborhood detail, call-type totals, district totals, and grand total.”

###### 8. Top 3 Selection

- “Top 3 per call type globally, or top 3 per call type within each district?”
- Expected Answer: “Top 3 per call type within each supervisor district.”

#### Step 2: High-Level Implementation Plan  
After clarifying requirements, here’s the overall approach we’ll follow:

Implementation Overview

##### Phase 1: Setup  
    → Import libraries, configure environment, create Spark session

##### Phase 2: Data Loading  
    → Define schema and load fire department data

##### Phase 3: Data Enrichment  
    → Parse dates, extract time dimensions, categorize call types

##### Phase 4: Weekly Aggregations  
    → Group by dimensions, calculate metrics (counts, averages, percentiles, stddev)

##### Phase 5: Comparative Analysis (Window Functions)  
    → Battalion comparison, moving averages, year-over-year growth

##### Phase 6: Efficiency Scoring & Ranking  
    → Create custom score, rank neighborhoods, filter top 3

##### Phase 7: Hierarchical Aggregation  
    → Apply rollup for multi-level reporting, assign performance tiers

##### Phase 8: Final Output  
    → Present results by tier, category, and district

![KEY TECHNIQUES USED IN THIS LAB](./key_techniques.png)

***

#### Phase 1: Import Pyspark Libraries and Create Spark Session

##### Step 3.1: Import PySpark Libraries 

In [ ]:

# import pyspark core modules
from pyspark import SparkContext
from  pyspark.sql import SparkSession
from  pyspark.sql.types import *

# import sql functions 
from  pyspark.sql.functions import (
    # basic column operations  
    col, lit, expr, when,

    # aggregations functions 
    sum, count, avg, min, max, stddev, 

    # Date/time functions 
    to_date, year, month, weekofyear,

    # Window functions 
    rank, dense_rank, row_number, percent_rank, lag, lead, 

    # Sorting 
    desc, asc 
)



# import dispaly utilities  
from IPython.display import display, clear_output
import pandas as pd 



print("-------- Libraries are imported successfuly ----------")

-------- Libraries are imported successfuly ----------


##### step 3.2: Configure Pandas Display Settings

Similar to SQK SET commands for output formatting.

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

print("Pandas display configured")

Pandas display configured


##### Step 3.3: Clean Previous Spark Session 

Prevent conflicts when re-running the notebook

In [ ]:

try : 
    spark.stop()
    print('Stopped previous spark session!!')
    print('Ready for new spark session!!')
    
except : 
    pass 


try : 
    sc = SparkContext._active_spark_context
    if sc : 
        sc.stop()
        print('stopped previous sparkContext !!')
except : 
    pass 


##### Step 3.4: Create Spark Session 

In [5]:
spark = SparkSession.builder\
    .appName('Lab_SF_business_problem_video_ 49')\
    .master('spark://spark-master:7077')\
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.debug.maxToStringFields', '1000')\
    .getOrCreate()

print("Spark started successfuly !!!")
print(f"Spark version : {spark.version}")
print(f"Spark Master : {spark.sparkContext.master}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 10:45:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark started successfuly !!!
Spark version : 3.5.1
Spark Master : spark://spark-master:7077


---

#### Phase 2: Data Loading 

##### What This Phase Does:

- Load CSV data with automatic schema inference

- Spark reads the header row and detects column types

- Verify data loaded successfully and preview schema

##### Why Schema Inference?

- Automatic column detection – matches CSV column order

- Type detection – Spark infers int, float, string types automatically

- Simpler code – no need to manually define 28 columns

- Flexible – works even if CSV column order changes

##### Key Columns We'll Use:

Neighborhood, Battalion, SupervisorDistrict – Geographic hierarchy

- CallType – Type of emergency

- CallDate – When the call occurred

- Delay – Response time (our key metric)

- Priority – Emergency priority level

##### SQL Equivalent:

-- Schema inference is like 

CREATE TABLE AS SELECT
CREATE TABLE fire_calls AS
SELECT * FROM csv_file('sf-fire-sample.csv', header=true, auto_detect=true);


In [6]:
# Define the schema 


# Set up the data source file's path 
sf_fire_file = '/opt/spark-data/sf-fire-calls.csv'

# load dataframe with San Francesco fire data 

fire_df = spark.read\
    .option('header', 'true')\
    .option('inferSchema', 'true')\
    .csv(sf_fire_file, samplingRatio= 0.00001)


fire_df = fire_df.withColumn("Delay", col("Delay").cast("float"))

                                                        
print(' Data is loaded successfully:')
print(f'Columns : {len(fire_df.columns)}')
print('Schema : ')
fire_df.printSchema()

print('Sample data : ')
pdf = fire_df.limit(5).toPandas()
display(pdf)

 Data is loaded successfully:
Columns : 28
Schema : 
root
 |-- CallNumber: integer (nullable = true)
 |-- UnitID: string (nullable = true)
 |-- IncidentNumber: integer (nullable = true)
 |-- CallType: string (nullable = true)
 |-- CallDate: string (nullable = true)
 |-- WatchDate: string (nullable = true)
 |-- CallFinalDisposition: string (nullable = true)
 |-- AvailableDtTm: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Battalion: string (nullable = true)
 |-- StationArea: integer (nullable = true)
 |-- Box: integer (nullable = true)
 |-- OriginalPriority: integer (nullable = true)
 |-- Priority: integer (nullable = true)
 |-- FinalPriority: integer (nullable = true)
 |-- ALSUnit: boolean (nullable = true)
 |-- CallTypeGroup: string (nullable = true)
 |-- NumAlarms: integer (nullable = true)
 |-- UnitType: string (nullable = true)
 |-- UnitSequenceInCallDispatch: integer (nullable = true

,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,Zipcode,Battalion,StationArea,Box,OriginalPriority,Priority,FinalPriority,ALSUnit,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay
0,20110016,T13,2003235,Structure Fire,01/11/2002,01/10/2002,Other,01/11/2002 01:51:44 AM,2000 Block of CALIFORNIA ST,SF,94109,B04,38,3362,3,3,3,False,None,1,TRUCK,2,4,5,Pacific Heights,"(37.7895840679362, -122.428071912459)",020110016-T13,2.950000
1,20110022,M17,2003241,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 03:01:18 AM,0 Block of SILVERVIEW DR,SF,94124,B10,42,6495,3,3,3,True,None,1,MEDIC,1,10,10,Bayview Hunters Point,"(37.7337623673897, -122.396113802632)",020110022-M17,4.700000
2,20110023,M41,2003242,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 02:39:50 AM,MARKET ST/MCALLISTER ST,SF,94102,B03,1,1455,3,3,3,True,None,1,MEDIC,2,3,6,Tenderloin,"(37.7811772186856, -122.411699931232)",020110023-M41,2.433333
3,20110032,E11,2003250,Vehicle Fire,01/11/2002,01/10/2002,Other,01/11/2002 04:16:46 AM,APPLETON AV/MISSION ST,SF,94110,B06,32,5626,3,3,3,False,None,1,ENGINE,1,6,9,Bernal Heights,"(37.7388432849018, -122.423948785199)",020110032-E11,1.500000
4,20110043,B04,2003259,Alarms,01/11/2002,01/10/2002,Other,01/11/2002 06:01:58 AM,1400 Block of SUTTER ST,SF,94109,B04,3,3223,3,3,3,False,None,1,CHIEF,2,4,2,Western Addition,"(37.7872890372638, -122.424236212664)",020110043-B04,3.483333


---

#### Phase 3 : Data Enrishment

##### What This Phase Does:

- Parse CallDate string to proper date type

- Extract time dimensions: year, month, week

- Categorize call types into broader groups (Medical, Fire, Alarm, Other)

##### Why This Matters:

- Time dimensions enable weekly aggregations and year-over-year comparisons

- Call type categories simplify analysis (100+ specific types → 4 categories)

- Proper date types enable date arithmetic and window functions

SQL Equivalent:


SELECT 
    
    *,    
    TO_DATE(CallDate, 'MM/dd/yyyy hh:mm:ss a') AS call_date,
    YEAR(call_date) AS call_year,
    MONTH(call_date) AS call_month,
    WEEK(call_date) AS call_week,
    CASE
        WHEN CallType LIKE '%Medical%' THEN 'Medical'
        WHEN CallType LIKE '%Fire%' THEN 'Fire'
        WHEN CallType LIKE '%Alarm%' THEN 'Alarm'
        ELSE 'Other'
    END AS call_type_category
FROM fire_calls;


##### Step 5.1: Parse Dates and Extract Time Dimensions 

In [7]:
enriched_df = fire_df\
    .withColumn('call_date', to_date('CallDate', 'MM/dd/yyyy'))\
    .withColumn('call_year', year('call_date'))\
    .withColumn('call_month', month('call_date'))\
    .withColumn('call_week', weekofyear('call_date'))

print("Added time columns : call_date, call_year, call_month, and call_week")
pdf = enriched_df.limit(5).toPandas()
display(pdf)

Added time columns : call_date, call_year, call_month, and call_week


,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,Zipcode,Battalion,StationArea,Box,OriginalPriority,Priority,FinalPriority,ALSUnit,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay,call_date,call_year,call_month,call_week
0,20110016,T13,2003235,Structure Fire,01/11/2002,01/10/2002,Other,01/11/2002 01:51:44 AM,2000 Block of CALIFORNIA ST,SF,94109,B04,38,3362,3,3,3,False,None,1,TRUCK,2,4,5,Pacific Heights,"(37.7895840679362, -122.428071912459)",020110016-T13,2.950000,2002-01-11,2002,1,2
1,20110022,M17,2003241,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 03:01:18 AM,0 Block of SILVERVIEW DR,SF,94124,B10,42,6495,3,3,3,True,None,1,MEDIC,1,10,10,Bayview Hunters Point,"(37.7337623673897, -122.396113802632)",020110022-M17,4.700000,2002-01-11,2002,1,2
2,20110023,M41,2003242,Medical Incident,01/11/2002,01/10/2002,Other,01/11/2002 02:39:50 AM,MARKET ST/MCALLISTER ST,SF,94102,B03,1,1455,3,3,3,True,None,1,MEDIC,2,3,6,Tenderloin,"(37.7811772186856, -122.411699931232)",020110023-M41,2.433333,2002-01-11,2002,1,2
3,20110032,E11,2003250,Vehicle Fire,01/11/2002,01/10/2002,Other,01/11/2002 04:16:46 AM,APPLETON AV/MISSION ST,SF,94110,B06,32,5626,3,3,3,False,None,1,ENGINE,1,6,9,Bernal Heights,"(37.7388432849018, -122.423948785199)",020110032-E11,1.500000,2002-01-11,2002,1,2
4,20110043,B04,2003259,Alarms,01/11/2002,01/10/2002,Other,01/11/2002 06:01:58 AM,1400 Block of SUTTER ST,SF,94109,B04,3,3223,3,3,3,False,None,1,CHIEF,2,4,2,Western Addition,"(37.7872890372638, -122.424236212664)",020110043-B04,3.483333,2002-01-11,2002,1,2


##### Step 5.2: Categorize Call Types

In [8]:
enriched_df = enriched_df\
    .withColumn(
        'call_type_category',
        when(col('CallType').contains('Medical'),'Medical')
        .when(col('CallType').contains('Fire'), 'Fire')
        .when(col('CallType').contains('Alarm'), 'Alarm')
        .otherwise('Other')
     )

print("Added call_type_categories column")

# preview categorization 

result= enriched_df.select('CallType', 'call_type_category')
pdf = result.limit(5).toPandas()
display(pdf)

Added call_type_categories column


,CallType,call_type_category
0,Structure Fire,Fire
1,Medical Incident,Medical
2,Medical Incident,Medical
3,Vehicle Fire,Fire
4,Alarms,Alarm


---

#### Phase 4 : Weekly Aggregations 

##### Step 6.1: Basic Aggregations (Count, Average)

##### What This Phase Does:  
Group data by multiple dimensions and calculate comprehensive metrics:

- Grouping: Neighborhood + Battalion + District + CallType + Year + Week

- Metrics: Counts, averages, percentiles (p25, p50, p75, p95), standard deviation, conditional counts

##### Why This Matters:

- Weekly granularity balances detail vs. noise (daily = too noisy, monthly = loses trends)

- Percentiles show distribution (median more robust than average for skewed data)

- Standard deviation measures consistency (critical for “efficiency”)

- Multi-dimensional grouping enables comparisons at different levels

##### Business Value:

- Count: Volume capacity (can handle high call volume?)

- Average: Central tendency of response time

- Percentiles: Understand outliers (95th percentile = worst-case scenarios)

- Std Dev: Consistency (low = predictable performance)

SQL Equivalent:


SELECT

    Neighborhood, Battalion, SupervisorDistrict,
    call_type_category, call_year, call_week,
    COUNT(*) AS weekly_calls,
    AVG(Delay) AS avg_delay,
    PERCENTILE_APPROX(Delay, 0.25) AS delay_p25,
    PERCENTILE_APPROX(Delay, 0.50) AS delay_median,
    PERCENTILE_APPROX(Delay, 0.75) AS delay_p75,
    PERCENTILE_APPROX(Delay, 0.95) AS delay_p95,
    STDDEV(Delay) AS delay_stddev,
    SUM(CASE WHEN Priority = '3' THEN 1 ELSE 0 END) AS high_priority_calls
FROM enriched_calls
GROUP BY Neighborhood, Battalion, SupervisorDistrict,
         call_type_category, call_year, call_week;


In [12]:
weekly_metrics = enriched_df.groupBy(
    'Neighborhood',
    'Battalion',
    'SupervisorDistrict',
    'call_type_category',
    'call_year',
    'call_week'
).agg(
    count('*').alias('weekly_calls'),
    avg('Delay').alias('avg_delay'),
    
)


print("Step 1 : Calculated basic aggregations (count, avg)")
print(f"Result : {weekly_metrics.count():,} rows")


result = weekly_metrics.orderBy('Weekly_calls')
pdf = result.limit(10).toPandas()
display(pdf)


Step 1 : Calculated basic aggregations (count, avg)
Result : 92,463 rows


,Neighborhood,Battalion,SupervisorDistrict,call_type_category,call_year,call_week,weekly_calls,avg_delay
0,Glen Park,B06,8,Alarm,2002,3,1,2.916667
1,South of Market,B03,6,Fire,2005,43,1,4.983333
2,Financial District/South Beach,B01,3,Other,2002,4,1,2.283333
3,Pacific Heights,B04,5,Medical,2005,43,1,4.133333
4,Chinatown,B01,3,Fire,2002,3,1,1.650000
5,Lakeshore,B08,7,Alarm,2005,42,1,2.300000
6,Lakeshore,B08,7,Alarm,2002,3,1,1.816667
7,Outer Mission,B09,11,Fire,2005,42,1,0.633333
8,Nob Hill,B01,3,Medical,2002,4,1,7.066667
9,Sunset/Parkside,B08,4,Alarm,2005,42,1,2.016667


##### Step 6.1: Add Percentile Calculattions 

![percentileVsPercentile_approx](./percentile.png )
 
##### Why We Choose PERCENTILE_APPROX: 




In [ ]:
# Exact percentile - NOT recommended for big data
# percentile(col, [0.25, 0.5, 0.75])  # Very expensive!

# Approximate percentile - RECOMMENDED ✓
expr("percentile_approx(Delay, 0.25)")  # Fast, scalable

# Can specify accuracy (default = 10000 = 99.9% accurate)
expr("percentile_approx(Delay, 0.25, 10000)")


##### Real-world example:

- Dataset: 1 billion emergency calls

- Exact percentile: 45 minutes, full cluster shuffle

- Approximate percentile: 3 minutes, 99.9% accurate

- Business decision: Same outcome, 15x faster!

In [13]:
# Part 2 :Add percentile to understand delay distribution 

weekly_metrics = enriched_df.groupBy(
    'Neighborhood',
    'Battalion',
    'SupervisorDistrict',
    'call_type_category',
    'call_year',
    'call_week'
).agg(
    count('*').alias('weekly_calls'),
    avg('Delay').alias('avg_delay'),
    expr('percentile_approx(Delay, 0.25)').alias('Delay_p25'),
    expr('percentile_approx(Delay, 0.5)').alias('Delay_median'),
    expr('percentile_approx(Delay, 0.75)').alias('Delay_p75'),
    expr('percentile_approx(Delay, 0.95)').alias('Delay_p95')
    
)

print("Step 2 : Added percentiles (p25, p50, p75, p95)")

# Preview percentile distribution 

result = weekly_metrics.select(
    'call_year',
    'Neighborhood',
    'call_type_category',
    'call_week',
    'weekly_calls',
    'avg_delay',
    'Delay_p25',
    'Delay_median',
    'Delay_p75',
    'Delay_p95',
).orderBy('call_year','call_week','Neighborhood','call_type_category', )

pdf = result.limit(10).toPandas()
display(pdf)

Step 2 : Added percentiles (p25, p50, p75, p95)


,call_year,Neighborhood,call_type_category,call_week,weekly_calls,avg_delay,Delay_p25,Delay_median,Delay_p75,Delay_p95
0,2000,Bayview Hunters Point,Alarm,16,1,4.316667,4.316667,4.316667,4.316667,4.316667
1,2000,Bayview Hunters Point,Alarm,17,4,3.204167,2.800000,2.866667,2.883333,4.266667
2,2000,Bayview Hunters Point,Alarm,18,1,2.166667,2.166667,2.166667,2.166667,2.166667
3,2000,Bayview Hunters Point,Alarm,24,1,2.850000,2.850000,2.850000,2.850000,2.850000
4,2000,Bayview Hunters Point,Alarm,26,1,1.966667,1.966667,1.966667,1.966667,1.966667
5,2000,Bayview Hunters Point,Alarm,33,1,2.150000,2.150000,2.150000,2.150000,2.150000
6,2000,Bayview Hunters Point,Alarm,41,1,2.133333,2.133333,2.133333,2.133333,2.133333
7,2000,Bayview Hunters Point,Alarm,42,1,3.366667,3.366667,3.366667,3.366667,3.366667
8,2000,Bayview Hunters Point,Alarm,43,1,4.033333,4.033333,4.033333,4.033333,4.033333
9,2000,Bayview Hunters Point,Alarm,44,1,1.800000,1.800000,1.800000,1.800000,1.800000


In [16]:
# Part 2 :Complete aggregation with all metrics

weekly_metrics = enriched_df.groupBy(
    'Neighborhood',
    'Battalion',
    'SupervisorDistrict',
    'call_type_category',
    'call_year',
    'call_week'
).agg(
    # basic metrics 
    count('*').alias('weekly_calls'),
    avg('Delay').alias('avg_delay'),

    # Distribution (percentiles)
    
    expr('percentile_approx(Delay, 0.25)').alias('Delay_p25'),
    expr('percentile_approx(Delay, 0.5)').alias('Delay_median'),
    expr('percentile_approx(Delay, 0.75)').alias('Delay_p75'),
    expr('percentile_approx(Delay, 0.95)').alias('Delay_p95'),
    
    # Consistency metrics
    stddev('Delay').alias('Delay_stddev'),

    # Conditional count (high priority Calls)

    sum(when(col("Priority") == '3', 1).otherwise(0)).alias('high_priority_calls')
    
)

result = weekly_metrics.select(
    # dimensions (time, georaphic, and call_category)
    'call_year', 
    'call_week',  
    'Neighborhood',   
    'call_type_category', 
    
    # measures 
    'weekly_calls',
    'high_priority_calls',
    'avg_delay',
    'Delay_stddev',
    'Delay_p25',
    'Delay_median',
    'Delay_p75',
    'Delay_p95',
).orderBy('call_year','Neighborhood','call_type_category', 'call_week')

pdf = result.limit(10).toPandas()
display(pdf)


print("Step 3 : Added stddev and conditional count")
print(f"Final Metrics : {weekly_metrics.count():,} rows")



,call_year,call_week,Neighborhood,call_type_category,weekly_calls,high_priority_calls,avg_delay,Delay_stddev,Delay_p25,Delay_median,Delay_p75,Delay_p95
0,2000,16,Bayview Hunters Point,Alarm,1,1,4.316667,NaN,4.316667,4.316667,4.316667,4.316667
1,2000,17,Bayview Hunters Point,Alarm,4,4,3.204167,0.709248,2.800000,2.866667,2.883333,4.266667
2,2000,18,Bayview Hunters Point,Alarm,1,1,2.166667,NaN,2.166667,2.166667,2.166667,2.166667
3,2000,24,Bayview Hunters Point,Alarm,1,1,2.850000,NaN,2.850000,2.850000,2.850000,2.850000
4,2000,26,Bayview Hunters Point,Alarm,1,1,1.966667,NaN,1.966667,1.966667,1.966667,1.966667
5,2000,33,Bayview Hunters Point,Alarm,1,1,2.150000,NaN,2.150000,2.150000,2.150000,2.150000
6,2000,41,Bayview Hunters Point,Alarm,1,1,2.133333,NaN,2.133333,2.133333,2.133333,2.133333
7,2000,42,Bayview Hunters Point,Alarm,1,1,3.366667,NaN,3.366667,3.366667,3.366667,3.366667
8,2000,43,Bayview Hunters Point,Alarm,1,1,4.033333,NaN,4.033333,4.033333,4.033333,4.033333
9,2000,44,Bayview Hunters Point,Alarm,1,1,1.800000,NaN,1.800000,1.800000,1.800000,1.800000


Step 3 : Added stddev and conditional count
Final Metrics : 92,463 rows


##### Step 6.4: Preview Weekly Metrics 

Semilar to SQL : 

SELECT 

    TOP 10  * 
FROM weekly_metrics 
ORDER BY WEEKLY_CALLS DESC


In [17]:
result = weekly_metrics.orderBy(desc('weekly_calls'))
pdf = result.limit(10).toPandas()

print('Top 10 neighborhoods by weekly call volume :')
display(pdf)

[Stage 71:=========================================>               (8 + 3) / 11]

Top 10 neighborhoods by weekly call volume :


,Neighborhood,Battalion,SupervisorDistrict,call_type_category,call_year,call_week,weekly_calls,avg_delay,Delay_p25,Delay_median,Delay_p75,Delay_p95,Delay_stddev,high_priority_calls
0,South of Market,B03,6,Medical,2018,22,22,3.327273,2.066667,2.883333,4.183333,5.850000,1.790832,13
1,South of Market,B03,6,Medical,2012,10,22,4.795455,2.766667,3.466667,4.500000,12.366667,4.651640,12
2,South of Market,B03,6,Medical,2018,37,21,4.107937,2.433333,3.366667,4.233333,4.550000,4.756302,11
3,Tenderloin,B02,6,Medical,2017,2,21,4.253175,2.516667,3.216667,5.083333,6.733333,3.451339,13
4,South of Market,B03,6,Medical,2014,18,20,4.970833,1.966667,2.833333,3.800000,17.299999,6.364613,14
5,Tenderloin,B03,6,Medical,2017,35,20,7.557500,2.333333,3.466667,4.666667,30.333334,11.230893,12
6,South of Market,B03,6,Medical,2014,9,19,3.943860,1.600000,3.083333,4.400000,15.566667,3.733462,7
7,South of Market,B03,6,Medical,2014,7,19,10.170176,2.166667,4.366667,11.700000,60.966667,14.586939,11
8,South of Market,B03,6,Medical,2016,14,19,4.257018,1.866667,3.016667,5.250000,16.950001,4.031302,9
9,South of Market,B03,6,Medical,2011,11,19,3.389474,2.083333,2.850000,3.283333,11.116667,2.224653,12


---

#### Phase 5: Comparative Analysis with Window Functions

##### What This Phase Does:

- Battalion comparison: How does each neighborhood compare to others in the same battalion?

- Moving averages: What’s the 7‑week trend (smooth out weekly noise)?

- Year‑over‑year: Is performance improving compared to last year?

##### Why Window Functions?:

- Window functions calculate values across related rows without collapsing groups.

- PARTITION BY = divide data into groups

- ORDER BY = order within groups

- OVER() = apply function across the window

- Unlike GROUP BY, window functions keep all original rows while adding comparative metrics.

##### Business Value:

- Battalion comparison: Fair comparison (same geographic/operational conditions)

- Moving averages: Identify trends vs. random weekly fluctuations

- YoY comparison: Track improvement initiatives, seasonal patterns

SQL Equivalent:

-- Battalion comparison



SELECT *,

    AVG(avg_delay) OVER (
        PARTITION BY Battalion, call_type_category, call_year
    ) AS battalion_avg_delay,

-- Moving average (7 weeks)

    AVG(avg_delay) OVER (
        PARTITION BY Neighborhood, call_type_category
        ORDER BY call_week
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS moving_avg_7w,

-- Year-over-year

    LAG(avg_delay, 1) OVER (
        PARTITION BY Neighborhood, call_type_category, call_week
        ORDER BY call_year
    ) AS prev_year_delay
FROM weekly_metrics;


##### Step 7.1: Define Battalion window  

In [18]:
from pyspark.sql import Window

# Define Window : group neighborhoods by their battallion
# This creates 'partitions' for fair comparison
battallion_window = Window.partitionBy(
    'Battalion',
    'call_type_category',
    'call_year',
)

print("Defined battallion window for comparison")
print("Partition by : Battalion, call_year ,and call_type_category")
print("Purpose : Compare neighborhoods within same battalion/conditions")

Defined battallion window for comparison
Partition by : Battalion, call_year ,and call_type_category
Purpose : Compare neighborhoods within same battalion/conditions


##### Step 7.2: Calculate Battalion Average

Apply window function to calculate average delay across battalion.

In [20]:
with_battalion_avg = weekly_metrics.withColumn(
    'battalion_avg_delay',
    avg('avg_delay').over(battallion_window)
)

print("Added battalion_avg_delay column")
print("each row now shows its battalion's average delay in a year")

Added battalion_avg_delay column
each row now shows its battalion's average delay in a year


##### Step 7.3: Calculate Differnce form Battalion Average

- Positive value = slower than battalion average.
- Negative = faster 

##### Step 7.4: Calculate 7-week Moving Average

Moving averages smooth out weekly noise to see true trends.

###### Similar to SQL: 

AVG(AVG_DELAY)  OVER(PARTITION BY NEIGHBORHOOD ORDER BY WEEK ROWS BETWEEN 6 PRECEIDING AND CURRENT ROW)

In [21]:
moving_avg_window = Window\
    .partitionBy('Neighborhood', 'call_type_category')\
    .orderBy('call_week')\
    .rowsBetween(-6, 0)

# Calculate 7 week rolling average 
with_moving_avg = with_battalion_avg.withColumn(
    'moving_avg_7w',
    avg('avg_delay').over(moving_avg_window)
)

print('Added 7-week moving average')

#Preview moving averages

result = with_moving_avg.select(
    'call_year','Battalion','Neighborhood','call_type_category', 'call_week', 'avg_delay', 'moving_avg_7w'
).limit(20).toPandas()

display(result)


Added 7-week moving average


,call_year,Battalion,Neighborhood,call_type_category,call_week,avg_delay,moving_avg_7w
0,2014,B10,Bayview Hunters Point,Alarm,1,3.016667,3.016667
1,2003,B10,Bayview Hunters Point,Alarm,1,6.400000,4.708333
2,2005,B10,Bayview Hunters Point,Alarm,1,2.516667,3.977778
3,2017,B10,Bayview Hunters Point,Alarm,1,3.133333,3.766667
4,2002,B10,Bayview Hunters Point,Alarm,1,2.733333,3.560000
5,2007,B10,Bayview Hunters Point,Alarm,1,5.066667,3.811111
6,2016,B10,Bayview Hunters Point,Alarm,1,3.016667,3.697619
7,2016,B10,Bayview Hunters Point,Alarm,2,3.425000,3.755952
8,2001,B10,Bayview Hunters Point,Alarm,2,3.641667,3.361905
9,2015,B10,Bayview Hunters Point,Alarm,2,5.200000,3.745238


##### Step 7.5: Year-over-Year Analysis - Define Window and Get Previous Year 


Compare same week this year vs last year to track imporvement.
Similat to SQL: LAG(avg_delay, 1) OVER(PARTITION BY Neighborhood, call_week ORDER BY call_year)

In [30]:
YoY_window = Window\
    .partitionBy('Neighborhood','call_type_category','call_week')\
    .orderBy('call_year')  

with_yoy= with_moving_avg.withColumn(
    'prev_year_delay', lag('avg_delay',1).over(YoY_window)
)

result = with_yoy.select(
    'Neighborhood','call_type_category', 'call_year', 'call_week', 'avg_delay', 'prev_year_delay'  
).orderBy('Neighborhood','call_type_category', 'call_year', 'call_week')

pdf = result.limit(10).toPandas()
display(pdf)

,Neighborhood,call_type_category,call_year,call_week,avg_delay,prev_year_delay
0,Bayview Hunters Point,Alarm,2000,16,4.316667,NaN
1,Bayview Hunters Point,Alarm,2000,17,3.204167,NaN
2,Bayview Hunters Point,Alarm,2000,18,2.166667,NaN
3,Bayview Hunters Point,Alarm,2000,24,2.850000,NaN
4,Bayview Hunters Point,Alarm,2000,26,1.966667,NaN
5,Bayview Hunters Point,Alarm,2000,33,2.150000,NaN
6,Bayview Hunters Point,Alarm,2000,41,2.133333,NaN
7,Bayview Hunters Point,Alarm,2000,42,3.366667,NaN
8,Bayview Hunters Point,Alarm,2000,43,4.033333,NaN
9,Bayview Hunters Point,Alarm,2000,44,1.800000,NaN


##### Step 7.6: Calculate the YoY Growth

Negataive growth : good imporvement (delays are decreasing)

In [35]:
with_yoy = with_yoy.withColumn(
    'yoy_growth',
    when(
        col('prev_year_delay').isNotNull(), 
        (((col('avg_delay')- col('prev_year_delay'))/ col('prev_year_delay'))*100)
    ).
    otherwise(None)
)



result = with_yoy.filter(col('prev_year_delay').isNotNull()).select(
    'Neighborhood','call_type_category', 'call_year', 'call_week', 'avg_delay', 'prev_year_delay', 'yoy_growth'
).orderBy('Neighborhood','call_type_category', 'call_year','call_week')

pdf = result.limit(10).toPandas()
display(pdf)



,Neighborhood,call_type_category,call_year,call_week,avg_delay,prev_year_delay,yoy_growth
0,Bayview Hunters Point,Alarm,2001,18,5.783333,2.166667,166.923066
1,Bayview Hunters Point,Alarm,2001,24,2.866667,2.850000,0.584795
2,Bayview Hunters Point,Alarm,2001,41,1.200000,2.133333,-43.750001
3,Bayview Hunters Point,Alarm,2001,44,2.683333,1.800000,49.074082
4,Bayview Hunters Point,Alarm,2001,48,1.100000,3.666667,-70.000000
5,Bayview Hunters Point,Alarm,2001,49,2.683333,3.266667,-17.857141
6,Bayview Hunters Point,Alarm,2002,16,1.966667,4.316667,-54.440153
7,Bayview Hunters Point,Alarm,2002,24,1.900000,2.866667,-33.720928
8,Bayview Hunters Point,Alarm,2002,26,2.183333,1.966667,11.016951
9,Bayview Hunters Point,Alarm,2002,27,2.533333,2.141667,18.287937
